In [1]:
import os, time, random

import numpy as np
import pandas as pd

import albumentations as A
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torch.optim as optim

from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold

import warnings

warnings.filterwarnings("ignore")



In [2]:
DIR_INPUT = "/kaggle/input/plant-pathology-2020-fgvc7"

SEED = 42
N_FOLDS = 5
N_EPOCHS = 3
BATCH_SIZE = 16
IMAGE_SIZE = (273, 409)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
device



device(type='cuda')

In [3]:
# Bugfix: original notebook had missing cell numbers; keep execution self-contained by defining dataframes here.
train_df = pd.read_csv(f"{DIR_INPUT}/train.csv")
test_df = pd.read_csv(f"{DIR_INPUT}/test.csv")
sample_sub = pd.read_csv(f"{DIR_INPUT}/sample_submission.csv")

TARGET_COLS = ["healthy", "multiple_diseases", "rust", "scab"]
assert all(c in train_df.columns for c in ["image_id"] + TARGET_COLS)
assert list(sample_sub.columns) == ["image_id"] + TARGET_COLS

# For StratifiedKFold, create a single-label strat target (argmax of one-hot labels).
train_df["strat_label"] = train_df[TARGET_COLS].values.argmax(axis=1)




In [4]:
class PlantDataset(Dataset):
    def __init__(self, df, transforms=None, test_set=False):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.test_set = test_set
        if not self.transforms:
            self.transforms = A.Compose([ToTensorV2(p=1.0)])

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, idx):
        image_src = DIR_INPUT + "/images/" + self.df.loc[idx, "image_id"] + ".jpg"
        image = cv2.imread(image_src, cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {image_src}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, IMAGE_SIZE)

        transformed = self.transforms(image=image)
        image = transformed["image"]

        if not self.test_set:
            labels = self.df.loc[idx, TARGET_COLS].values.astype(np.float32)
            labels = torch.from_numpy(labels)
            return image, labels
        else:
            return image




In [5]:
# Keep augmentation minimal and consistent with original intent (Normalize + ToTensor).
transforms_train = A.Compose(
    [
        A.Normalize(p=1.0),
        ToTensorV2(p=1.0),
    ]
)

transforms_valid = A.Compose(
    [
        A.Normalize(p=1.0),
        ToTensorV2(p=1.0),
    ]
)




In [6]:
class PlantModel(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        # Bugfix for torchvision>=0.13: pretrained=True is deprecated; use weights for compatibility.
        self.backbone = torchvision.models.resnet50(
            weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1
        )

        in_features = self.backbone.fc.in_features
        self.logit = nn.Linear(in_features, num_classes)

    def forward(self, x):
        batch_size, C, H, W = x.shape

        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)

        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)

        x = F.adaptive_avg_pool2d(x, 1).reshape(batch_size, -1)
        x = F.dropout(x, 0.25, self.training)
        x = self.logit(x)
        return x




In [7]:
# Training/inference helpers (minimal additions to replace missing external .pth files).
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running = 0.0
    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        running += loss.item() * images.size(0)
    return running / len(loader.dataset)


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    probs_all = []
    for images in loader:
        images = images.to(device)
        logits = model(images)
        # Bugfix: softmax must specify dim for multi-class probabilities.
        probs = F.softmax(logits, dim=1)
        probs_all.append(probs.detach().cpu().numpy())
    return np.concatenate(probs_all, axis=0)




In [8]:
dataset_test = PlantDataset(df=test_df, test_set=True, transforms=transforms_valid)
testloader = DataLoader(
    dataset_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)



In [9]:
# Minimal CV training to produce fold models locally and enable submission generation.
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

all_test_probs = []

start = time.perf_counter()
for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["strat_label"])):
    tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
    va_df = train_df.iloc[va_idx].reset_index(drop=True)

    train_ds = PlantDataset(tr_df, transforms=transforms_train, test_set=False)
    valid_ds = PlantDataset(va_df, transforms=transforms_valid, test_set=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    model = PlantModel(num_classes=4).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(N_EPOCHS):
        _ = train_one_epoch(model, train_loader, optimizer, criterion)

    # Convert logits to class probabilities (4 columns) for submission.
    test_probs_fold = predict_proba(model, testloader)
    all_test_probs.append(test_probs_fold)

elapsed = time.perf_counter() - start
print(f"Finished Training+Inference in {elapsed:.2f} seconds")



Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 116MB/s]


Finished Training+Inference in 128.89 seconds


In [10]:
# Bugfix: average across folds correctly to shape (n_test, 4).
test_probs_mean = np.mean(np.stack(all_test_probs, axis=0), axis=0)
test_probs_mean.shape



(183, 4)

In [11]:
# Bugfix: ensure correct column assignment with matching shape and correct row order.
submission_df = pd.read_csv(f"{DIR_INPUT}/sample_submission.csv")
submission_df[TARGET_COLS] = test_probs_mean.astype(np.float32)

# Safety: clip to valid probability range.
submission_df[TARGET_COLS] = submission_df[TARGET_COLS].clip(0.0, 1.0)

submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Wrote {submission_path} with shape {submission_df.shape}")



Wrote submission.csv with shape (183, 5)


In [12]:
submission_df.head()

,image_id,healthy,multiple_diseases,rust,scab
0,Test_0,1.383842e-04,5.437189e-04,0.000111,9.992070e-01
1,Test_1,6.048653e-04,5.817655e-04,0.000280,9.985337e-01
2,Test_2,1.744355e-04,1.007493e-04,0.999571,1.542053e-04
3,Test_3,5.856553e-08,2.762605e-07,1.000000,1.000031e-07
4,Test_4,1.798723e-07,7.716805e-07,0.999999,4.819918e-07
